# AM Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enph-479-edge-ai-stt/enph-479-edge-ai-stt/blob/feat/streamline-am-training/training/notebooks/am_training.ipynb)

Trains the acoustic model (3x256 unidirectional LSTM + CTC) on LibriSpeech train-clean-100 and prints greedy dev-clean CER after every epoch. **Runtime: T4 GPU.**

- **Checkpoints go to Drive** (`MyDrive/enph479-stt/runs/<RUN_NAME>/`): `latest.pt` after every epoch, `best.pt` when dev CER improves. If the session dies, reconnect and run all cells. Training resumes from the last finished epoch.
- **No feature cache.** Audio is downloaded to the VM's scratch disk each session and features are computed on the fly in the DataLoader workers. If GPU utilization in Colab's Resources panel stays low while the CPU is pegged, feature extraction is the bottleneck.
- Specs (30-symbol vocab, feature constants) are provisional until `shared/specs` is frozen.

## 1. Pull the repo

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/enph-479-edge-ai-stt/enph-479-edge-ai-stt.git"
REPO_DIR = "/content/enph-479-edge-ai-stt"
BRANCH = "feat/streamline-am-training"  # TODO: back to "main" (and the badge) before the PR

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
# Always land on the tip of BRANCH, even if an earlier run left the clone on another branch.
git = ["git", "-C", REPO_DIR]
subprocess.run([*git, "fetch", "origin", BRANCH], check=True)
subprocess.run([*git, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run([*git, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Installs this checkout's package and its dependencies (jiwer, soundfile);
# torch and torchaudio come with Colab.
training_dir = os.path.join(REPO_DIR, "training")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", training_dir], check=True)

# The editable install only takes effect at interpreter start, so put src/ on the
# path for this already-running kernel.
sys.path.insert(0, os.path.join(training_dir, "src"))
importlib.invalidate_caches()

## 2. Configure the run

`TRAIN_EVERY = 10` trains on every 10th train-clean-100 utterance (~10 h, all speakers) as a test run; `1` uses all 100 h. `RUN_NAME` is the run's identity: the same name resumes, a new name starts fresh.

In [ ]:
import torch
from google.colab import drive

from training.am.train import TrainConfig

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
drive.mount("/content/drive")

TRAIN_EVERY = 10
RUN_NAME = f"am-3x256-every{TRAIN_EVERY}"
cfg = TrainConfig(run_dir=f"/content/drive/MyDrive/enph479-stt/runs/{RUN_NAME}", epochs=20)
cfg

## 3. Data

Downloads train-clean-100 (6.3 GB) and dev-clean (337 MB) to `/content/data` (skipped if already there in this session), then computes CMVN from the training utterances.

In [ ]:
from training.am.dataset import compute_cmvn_over, list_utterances
from training.data.librispeech import download_subset

train_items = list_utterances(download_subset("train-clean-100", "/content/data"))[::TRAIN_EVERY]
dev_items = list_utterances(download_subset("dev-clean", "/content/data"))
print(f"train {len(train_items)} utts, dev {len(dev_items)} utts")

# Every 10th training utterance is plenty for a global mean/std.
mean, std = compute_cmvn_over(train_items[::10], num_workers=cfg.num_workers)

## 4. Train

Expect the blank wall first: dev CER sits near 1.0 while the model outputs only blanks, then characters appear and it falls. Progress lines show s/batch; each epoch line shows train and eval time.

In [ ]:
from training.am.train import train

best_cer = train(cfg, train_items, dev_items, mean, std)
print(f"best dev CER {best_cer:.4f}")